# Runnables

## Goal 

understandingL:

- What a Runnables actually is in LangChain
- Why LangChain introduced the Runnables abstraction.
- What problem Runnables solve.
- Why different LangChain components can be composed together.
- The common interface shared by Runnables.
- How `invoke()`, `batch()`, `stream()` and `ainvoke()` relate to the Runnable interface.
- The different between a Runnable and a RunnableSequence.
- How different Runnable types can be combined to build production pipelines.
- How Runnables will become the foundation for more advanced concepts such as tools, agents and LangGraph.

In [1]:
# Load the environment
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# Read the model name
import os
MODEL_NAME = os.environ["GEMINI_MODEL"]
API_KEY = os.environ["GOOGLE_GENERATIVE_AI_API_KEY"]

In [3]:
# Create the LangChain model
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI (model = MODEL_NAME, api_key =API_KEY, temperature = 0)

In [4]:
# Create a Runnable from a normal function
from langchain_core.runnables import RunnableLambda

def double(x):
    return x * 2

double_runnable = RunnableLambda(double)

In [5]:
# Invoke the new runnable function
double_result = double_runnable.invoke(5)
print(double_result)
type(double_runnable)

10


langchain_core.runnables.base.RunnableLambda

In [27]:
# Define the parser
from langchain_core.output_parsers import StrOutputParser
str_parser = StrOutputParser()

In [7]:
# Create the function then create the RUnnable
def add_prefix(text):
    return "AI: " + text

prefix_runnable = RunnableLambda(add_prefix)


In [ ]:
# Create and invoke the chain
chain = str_parser | prefix_runnable

print(chain.invoke("LangChain is useful."))

AI: LangChain is useful.


In [9]:
# batch with RunnableLambda

def double (x):
    return x * 2

double_runnable = RunnableLambda(double)

results = double_runnable.batch([1,2,3,4,5,6])

print(type(results))
print(results)

print(type(double_runnable))

<class 'list'>
[2, 4, 6, 8, 10, 12]
<class 'langchain_core.runnables.base.RunnableLambda'>


In [10]:
# Chunk with RunnableLambda
for chunk in double_runnable.stream(5):
    print(type(chunk))
    print(repr(chunk))

<class 'int'>
10


In [11]:
def introduce (data):
    return f"{data['name']} is {data['age']} years old."


In [12]:
introduce_runnable = RunnableLambda(introduce)

result = introduce_runnable.invoke(
    {
        "name": "Mohamed",
        "age": 30
    }
)

print(result)
print(type(result))

Mohamed is 30 years old.
<class 'str'>


In [13]:
# async RunnableLambda

async def double_async(x):
    return x * 2

double_runnable_async = RunnableLambda(double_async)

result = await double_runnable_async.ainvoke(5)

print(result)
print(type(double_runnable_async))

10
<class 'langchain_core.runnables.base.RunnableLambda'>


In [14]:
# RunnablePassthrough

from langchain_core.runnables import RunnablePassthrough

passthrough = RunnablePassthrough() # Receive an input and pass that same input through unchanged.

result = passthrough.invoke("Hello")

print (result)

Hello


In [15]:
result = passthrough.invoke(
    {
        "question": "What is LangChain?",
        "topic":"AI"
    }
)

print(type(result))
print(result)

<class 'dict'>
{'question': 'What is LangChain?', 'topic': 'AI'}


In [ ]:
# RunnableParallel

from langchain_core.runnables import RunnableParallel
def fake_retrieve(question):
    return["LangChain is a framework "+ question]

retriever = RunnableLambda(fake_retrieve)

chain = RunnableParallel( # RunnableParallel combines branch outputs into one dictionary
    {
        "question": RunnablePassthrough(), # When invoke it will return the same input inside the invoke
        "context": retriever
    }
)

In [17]:
result = chain.invoke("What is LangChain?")
print(result)

{'question': 'What is LangChain?', 'context': ['Document about What is LangChain?']}


```text
                  "What is LangChain?"
                         │
              ┌──────────┴──────────┐
              ↓                     ↓
 RunnablePassthrough          retriever
              ↓                     ↓
          question              context
              │                     │
              └──────────┬──────────┘
                         ↓
                    final dict
```

In [18]:
def add_ten(x):
    return x +10

add_ten_runnable = RunnableLambda(add_ten)

parallel = RunnableParallel(
    {
        "original":RunnablePassthrough(),
        "doubled":RunnableLambda(double),
        "plus_ten": add_ten_runnable
    }
)

In [19]:
parallel.invoke(5)

{'original': 5, 'doubled': 10, 'plus_ten': 15}

## Mini prompt pipeline

In [20]:
# Create the prompt
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer the question using the provided context."),
        ("human", "Question: {question}\nContext: {context}")
    ]
)

In [22]:
# Create the context_chain
context_chain = {
    "question": RunnablePassthrough(),
    "context": retriever,
}

In [24]:
# Create the chain

rag_input = context_chain | prompt

In [25]:
result = rag_input.invoke("What is LangChain?")

print(type(rag_input))
print(type(result))
print(result)

<class 'langchain_core.runnables.base.RunnableSequence'>
<class 'langchain_core.prompt_values.ChatPromptValue'>
messages=[SystemMessage(content='Answer the question using the provided context.', additional_kwargs={}, response_metadata={}), HumanMessage(content="Question: What is LangChain?\nContext: ['Document about What is LangChain?']", additional_kwargs={}, response_metadata={})]


The current pipeline"

```text
"What is LangChain?"
        ↓
RunnableParallel
        ↓
{
    question: "What is LangChain?",
    context: [...]
}
        ↓
ChatPromptTemplate
        ↓
ChatPromptValue
```

In [28]:
# Create the generation chain

generation_chain = rag_input | llm

# Create the full chain

full_chain= generation_chain | str_parser

In [29]:
print(type(full_chain))
result = full_chain.invoke("What is LangChain?")
print(type(result))

<class 'langchain_core.runnables.base.RunnableSequence'>
<class 'langchain_core.messages.base.TextAccessor'>


In [30]:
print(result)

Based on the provided context, it is simply referred to as a "Document about What is LangChain?" without any further details.
